In [1]:
import pandas as pd
import pickle

jobs = pd.read_pickle(
    "jobs.pkl"
)

with open(
    "job_embeddings.pkl",
    "rb"
) as f:
    job_embeddings = pickle.load(f)

print(jobs.shape)
print(job_embeddings.shape)

(10000, 10)
(10000, 384)


In [6]:
import os

print(os.getcwd())

d:\job_recommendation\notebooks


In [7]:
import pdfplumber

resume_text = ""

with pdfplumber.open(
    "../Data_Sceince_Resume.pdf"
) as pdf:

    for page in pdf.pages:

        text = page.extract_text()

        if text:
            resume_text += text

print(
    resume_text[:1000]
)

Chandana Das
Adm. No. 21JE1089 (cid:131)6294091002
#21je1089@iitism.ac.in (cid:239)linkedin §github
Education
IITISMDHANBAD OCT2022–May2026
BacheloroftechnologyinEngineeringPhysics Dhanbad,Jharkhand
Projects
AzureRetailDataEngineeringPipeline|AzureDataLake,PySpark,SQL,PowerBI|Github
• Builtanend-to-endAzure-inspireddataengineeringpipelineprocessing100K+e-commercetransactionsthrough
Bronze–Silver–Goldarchitecture,implementingdataprofiling,transformation,andanalytics-readydatamodeling.
• EngineeredETLworkflowsusingPython,SQL,Parquet,andPowerBItocreatecustomer,seller,product,andsales
performancedatamarts,enablingKPItrackingandbusinessintelligencereporting.
AIPoweredJobRecommendationSystem|Python,Pandas,Scikit-learn,Streamlit|Github
• DevelopedanAI-poweredjobrecommendationplatformthatautomaticallyextractsskillsfromresumesandmatches
candidateswithrelevantopportunitiesusingNLP,SentenceTransformers,andsemanticsimilaritytechniques.
• BuiltanddeployedaninteractiveStreamlitapplicationprocessingl

In [8]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

resume_embedding = model.encode(
    resume_text,
    convert_to_numpy=True
)

print(resume_embedding.shape)

c:\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


(384,)


In [9]:
from sklearn.metrics.pairwise import cosine_similarity

similarities = cosine_similarity(
    [resume_embedding],
    job_embeddings
).flatten()

print(similarities.shape)

(10000,)


In [10]:
import numpy as np

top_indices = np.argsort(
    similarities
)[-20:][::-1]

recommended_jobs = jobs.iloc[
    top_indices
].copy()

recommended_jobs["match_score"] = (
    similarities[top_indices]
    * 100
)

recommended_jobs[
    [
        "title",
        "company_name",
        "location",
        "match_score"
    ]
]

,title,company_name,location,match_score
235,Senior Power BI Developer,Applicantz,United States,54.158340
187,Informatica Cloud Migration Lead-24-00289,"Datasoft Technologies, Inc.","Columbus, OH",52.833782
7492,Lead Data Scientist,Intellisoft Technologies,"Virginia Beach, VA",52.001495
5583,Data Scientist - Databricks,Techions,"Santa Clara, CA",51.518124
1005,Research And Development Specialist,Net2Source Inc.,"Mossville, IL",51.389851
1509,Data Engineer - Full Time,"Computer Data Concepts, Inc","Kansas City, MO",51.208961
7771,Generative AI - Data Scientist,ChabezTech LLC,"Los Angeles, CA",50.688206
252,Sr. Azure Databricks Engineer,Sunray Informatics Inc,"Summit, NJ",50.230343
656,Technical Lead,CodeMonk,United States,50.195980
1707,Data Architect,NLB Services,"Dallas, TX",50.020527


In [11]:
titles = recommended_jobs["title"]

titles.tolist()

['Senior Power BI Developer',
 ' Informatica Cloud Migration Lead-24-00289',
 'Lead Data Scientist',
 'Data Scientist - Databricks',
 'Research And Development Specialist',
 'Data Engineer - Full Time',
 'Generative AI - Data Scientist',
 'Sr. Azure Databricks Engineer',
 'Technical Lead',
 'Data Architect',
 'Data Engineer 3: 24-00989',
 'Azure Synapse Data Engineer',
 'MS Power Apps/Dynamics Software Engineer',
 'Data Science Intern DIN48',
 'Junior Data Analytics Engineer',
 'Software Engineer II, Data',
 'Program Analyst Power BI',
 'Cloud Engineer (Azure)',
 'Data Analyst',
 'Data Engineer']

In [15]:
top_indices = np.argsort(similarities)[-100:][::-1]

top_jobs = jobs.iloc[top_indices]

In [16]:
top_jobs['title']

235                      Senior Power BI Developer
187      Informatica Cloud Migration Lead-24-00289
7492                           Lead Data Scientist
5583                   Data Scientist - Databricks
1005           Research And Development Specialist
                           ...                    
8185                                 Data Engineer
8254               Strategic Operations Supervisor
5098          Logistics and Transportation Manager
7986    Geismar Quality Lab Technician #: 24-02677
9390            Regional Sales Manager - Northwest
Name: title, Length: 100, dtype: object

In [17]:
title_embeddings = model.encode(
    jobs['title'].tolist()
)

In [27]:
from sklearn.cluster import KMeans

kmeans = KMeans(
    n_clusters=50,
    random_state=42
)

jobs['cluster'] = kmeans.fit_predict(
    job_embeddings
)

In [28]:
jobs['cluster'].value_counts()

cluster
13    326
9     308
10    304
21    288
27    288
43    275
3     274
16    273
5     269
15    264
41    262
1     250
20    235
22    234
30    224
12    224
8     223
18    223
32    222
48    219
38    219
0     214
49    210
44    208
31    193
7     192
35    191
19    191
2     190
23    189
25    189
45    188
17    184
24    184
26    179
28    174
42    169
37    168
46    168
47    160
4     159
11    156
39    129
40    119
14    118
34    112
6      95
29     87
36     53
33     27
Name: count, dtype: int64

In [29]:
role_counts = (
    top_jobs['title']
    .value_counts()
    .head(10)
)

print(role_counts)

title
Data Engineer                                 4
Data Analyst                                  2
Data Architect                                2
Data Scientist - Databricks                   1
Research And Development Specialist           1
 Informatica Cloud Migration Lead-24-00289    1
Senior Power BI Developer                     1
Generative AI - Data Scientist                1
Sr. Azure Databricks Engineer                 1
Technical Lead                                1
Name: count, dtype: int64


In [30]:
jobs['cluster'] = kmeans.labels_

jobs.groupby('cluster')['title'].count().sort_values(
    ascending=False
).head(20)

cluster
13    326
9     308
10    304
21    288
27    288
43    275
3     274
16    273
5     269
15    264
41    262
1     250
20    235
22    234
12    224
30    224
18    223
8     223
32    222
48    219
Name: title, dtype: int64

In [32]:
for c in range(20):

    print("\n")
    print("="*60)

    print(
        f"Cluster {c}"
    )

    sample_titles = (
        jobs[
            jobs['cluster']==c
        ]['title']
        .head(15)
        .tolist()
    )

    for title in sample_titles:

        print(title)



Cluster 0
Copywriter
Marketing Intern
Marketing Assistant
Digital Marketing Manager
Social Media Marketing Specialist
Senior Copywriter
Customer Service Events Representative
Brand Director
Content Writer
Director of Paid Advertising (Legal)
Digital Marketing Assistant
Senior Editor, GOAL US
Marketing Specialist 
Vice President, Marketing and Communications
TV News MMJ


Cluster 1
Field Technician, Locator
Maintenance Technician III - Multiple Shifts
Field Service Technician
Hydroblaster/ Field Technician
PROGRAM TECHNICIAN III
Facilities Engineer
Maintenance Supervisor
Test Technician I - 2nd Shift (Onsite)
Fire Alarm Designer
HVACR Service Technician
Maintenance Mechanic Specialist I
Facilities Maintenance Technician - DIA - $32.65 Per Hour
Maintenance E&I Technician
Maintenance Tech  A
Automotive Technician


Cluster 2
Oracle Database Administrator
LOCAL to AZ / QA Automation Engineer/SDET  USC GC ONLY
Automation Tester
Technical Writer
R Developer
Senior Oracle Database Administr

In [33]:
resume_embedding

array([-8.16603452e-02, -6.28464371e-02, -9.37961563e-02,  5.85461967e-04,
       -4.19471301e-02, -9.56152976e-02,  3.36644128e-02,  6.48571644e-03,
       -1.28980622e-01,  4.16315719e-02,  4.61790711e-02, -8.82738233e-02,
        2.15872452e-02, -3.68853323e-02, -1.60519183e-02,  8.05424675e-02,
        5.61427884e-02, -4.46128063e-02, -9.39429924e-02, -7.75346532e-02,
        7.40265250e-02,  1.42501462e-02, -4.34838943e-02, -6.61085397e-02,
        2.55880877e-02,  2.05920264e-03, -6.92962185e-02, -9.39780381e-03,
       -9.04162880e-05, -8.02150667e-02, -4.97258827e-02,  2.35732775e-02,
       -8.05698987e-03,  7.50571713e-02,  8.64200071e-02,  4.56062704e-02,
        2.37047132e-02,  1.45191234e-02, -1.70047060e-02, -2.04119962e-02,
        2.73716431e-02, -6.24442175e-02, -3.24609801e-02, -2.66344305e-02,
       -5.35509512e-02,  6.40511699e-03, -3.24600749e-02, -7.37830475e-02,
       -3.17503363e-02,  3.97878289e-02, -9.93447602e-02,  2.70264018e-02,
       -6.13878993e-03,  

In [34]:
top_jobs

,title,description,company_name,location,formatted_experience_level,formatted_work_type,work_type,description_length,job_text,job_text_length
235,Senior Power BI Developer,"VISA SPONSORSHIP IS NOT AVAILABLE.\nOur large,...",Applicantz,United States,Mid-Senior level,Contract,CONTRACT,1927,Senior Power BI Developer VISA SPONSORSHIP IS ...,1953
187,Informatica Cloud Migration Lead-24-00289,Informatica Cloud Migration LeadOnsiteAbout th...,"Datasoft Technologies, Inc.","Columbus, OH",Mid-Senior level,Contract,CONTRACT,1800,Informatica Cloud Migration Lead-24-00289 Inf...,1843
7492,Lead Data Scientist,Company DescriptionIntellisoft Technologies is...,Intellisoft Technologies,"Virginia Beach, VA",Not Specified,Contract,CONTRACT,2300,Lead Data Scientist Company DescriptionIntelli...,2320
5583,Data Scientist - Databricks,"At Techions, we pride ourselves on being at th...",Techions,"Santa Clara, CA",Not Specified,Full-time,FULL_TIME,3348,"Data Scientist - Databricks At Techions, we pr...",3028
1005,Research And Development Specialist,Job Title: Development/Research Engineer Locat...,Net2Source Inc.,"Mossville, IL",Associate,Contract,CONTRACT,2356,Research And Development Specialist Job Title:...,2392
...,...,...,...,...,...,...,...,...,...,...
8185,Data Engineer,We are actively seeking a Financial Data Engin...,Unknown Company,New York City Metropolitan Area,Not Specified,Full-time,FULL_TIME,1569,Data Engineer We are actively seeking a Financ...,1583
8254,Strategic Operations Supervisor,\nJob Description \n\n** Please note; an activ...,"BAE Systems, Inc.","Nashua, NH",Mid-Senior level,Full-time,FULL_TIME,5161,Strategic Operations Supervisor \nJob Descript...,3032
5098,Logistics and Transportation Manager,REQ ID – 32309173ROLE – Logistics and Transpor...,WalkWater Technologies,"Austin, TX",Mid-Senior level,Contract,CONTRACT,1033,Logistics and Transportation Manager REQ ID – ...,1070
7986,Geismar Quality Lab Technician #: 24-02677,Job Title: Geismar Quality Lab Technician\n\nL...,HireTalent - Diversity Staffing & Recruiting Firm,"Geismar, LA",Mid-Senior level,Contract,CONTRACT,4317,Geismar Quality Lab Technician #: 24-02677 Job...,3043


In [36]:
print(jobs.columns)

Index(['title', 'description', 'company_name', 'location',
       'formatted_experience_level', 'formatted_work_type', 'work_type',
       'description_length', 'job_text', 'job_text_length', 'cluster'],
      dtype='object')


In [37]:
print(top_jobs.columns)

Index(['title', 'description', 'company_name', 'location',
       'formatted_experience_level', 'formatted_work_type', 'work_type',
       'description_length', 'job_text', 'job_text_length'],
      dtype='object')


In [39]:
jobs['cluster'] = kmeans.fit_predict(job_embeddings)

In [40]:
jobs.to_pickle("clustered_jobs.pkl")

In [41]:
jobs = pd.read_pickle("clustered_jobs.pkl")

In [42]:
print(jobs.columns)

Index(['title', 'description', 'company_name', 'location',
       'formatted_experience_level', 'formatted_work_type', 'work_type',
       'description_length', 'job_text', 'job_text_length', 'cluster'],
      dtype='object')


In [43]:
top_indices = np.argsort(
    similarities
)[-100:][::-1]

top_jobs = jobs.iloc[top_indices].copy()

In [44]:
top_jobs['cluster'].value_counts()

cluster
26    42
2     15
20    10
30     7
8      4
46     4
44     3
5      3
27     3
3      2
28     2
24     1
4      1
10     1
25     1
23     1
Name: count, dtype: int64

In [45]:
top_jobs[
    top_jobs['cluster'] == 26
][['title']].head(30)

,title
235,Senior Power BI Developer
187,Informatica Cloud Migration Lead-24-00289
7492,Lead Data Scientist
5583,Data Scientist - Databricks
1509,Data Engineer - Full Time
7771,Generative AI - Data Scientist
252,Sr. Azure Databricks Engineer
656,Technical Lead
1707,Data Architect
940,Data Engineer 3: 24-00989


In [46]:
top_jobs[
    top_jobs['cluster'] == 2
][['title']].head(30)

,title
9129,Data Systems Engineer - Onsite
4654,SAP System Analyst
6808,Senior Quality Assurance Analyst-ProdDev Bay Area
7988,Oracle Fusion Cloud (Finance) Developer
8219,Senior Technical Lead - Investment Management
2881,Production Support Engineer
5136,Summer IT Intern - Identity & Access Mgmt Engi...
3637,Sr. Application Developer
2514,SAP FICA with Utilities- Functional
1369,SDET/QA Automation Engineer


In [47]:
top_jobs[
    top_jobs['cluster'] == 20
][['title']].head(30)

,title
2503,Cloud Engineer (Azure)
7859,Data Architect
5633,Senior Data Engineer
4888,Data Lake Architect @ ONSITE
7913,Sr. Software Engineer
3633,Junior Data Analyst
4845,Snowflake/AWS Integration Architect
738,Migration Architect
1269,Azure Migration Engineer
3597,Hiring for Back end engineers/C# .NET stack ||...


In [48]:
top_jobs[top_jobs['cluster']==26]['title']

235                             Senior Power BI Developer
187             Informatica Cloud Migration Lead-24-00289
7492                                  Lead Data Scientist
5583                          Data Scientist - Databricks
1509                            Data Engineer - Full Time
7771                       Generative AI - Data Scientist
252                         Sr. Azure Databricks Engineer
656                                        Technical Lead
1707                                       Data Architect
940                             Data Engineer 3: 24-00989
9801                          Azure Synapse Data Engineer
6774             MS Power Apps/Dynamics Software Engineer
5728                            Data Science Intern DIN48
4254                       Junior Data Analytics Engineer
3126                           Software Engineer II, Data
1590                             Program Analyst Power BI
5422                                         Data Analyst
1891          

In [49]:
career_domains = [
    "Data Engineering",
    "Data Science",
    "Business Intelligence",
    "Data Analytics"
]

for i, domain in enumerate(career_domains, start=1):
    print(f"{i}. {domain}")

1. Data Engineering
2. Data Science
3. Business Intelligence
4. Data Analytics
